# Phase 3 & 4: Churn Labeling & XGBoost Predictive Modeling
### Target: Predict Customer Churn Probability in Next 90 Days

**Methodology:**
- **Observation Period:** Dec 1, 2009 to Aug 31, 2011 (extract behavioral features)
- **Prediction Window:** Sep 1, 2011 to Dec 9, 2011 (90-day future horizon)
- **Target Definition:** `Churn = 1` if customer made 0 purchases in prediction window; `Churn = 0` if they repurchased

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

# 1. Load clean transaction data
df = pd.read_parquet(r"d:\data_proj\data\processed\clean_retail.parquet")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Loaded {len(df):,} transactions!")

# 2. Split into Observation vs Prediction window (Cutoff: Sep 1, 2011)
cutoff_date = pd.to_datetime('2011-09-01')
obs_df = df[df['InvoiceDate'] < cutoff_date]
pred_df = df[df['InvoiceDate'] >= cutoff_date]

print(f"Observation window transactions: {len(obs_df):,}")
print(f"Prediction window transactions:  {len(pred_df):,}")

### Step 2: Feature Engineering (12 Behavioral Predictors)

In [2]:
# Invoice-level spend
invoice_totals = obs_df.groupby(['Customer ID', 'Invoice'])['TotalPrice'].sum().reset_index()

# Core Customer RFM & Tenure Features
features = obs_df.groupby('Customer ID').agg(
    recency=('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    frequency=('Invoice', 'nunique'),
    monetary=('TotalPrice', 'sum'),
    total_items=('Quantity', 'sum'),
    unique_products=('StockCode', 'nunique'),
    customer_tenure_days=('InvoiceDate', lambda x: max((cutoff_date - x.min()).days, 1))
).reset_index()

# Basket-Level Metrics
basket_stats = invoice_totals.groupby('Customer ID')['TotalPrice'].agg(
    avg_order_value='mean',
    max_order_value='max',
    std_order_value='std'
).reset_index().fillna(0)

features = features.merge(basket_stats, on='Customer ID', how='left')

# Purchase Rates & Velocities
features['avg_items_per_order'] = (features['total_items'] / features['frequency']).round(1)
features['purchase_frequency_rate'] = (features['frequency'] / (features['customer_tenure_days'] / 30.4)).round(2)
features['monthly_spend_rate'] = (features['monetary'] / (features['customer_tenure_days'] / 30.4)).round(2)

# Create Churn Target (Did customer buy after Sep 1, 2011?)
future_customers = set(pred_df['Customer ID'].unique())
features['churn'] = features['Customer ID'].apply(lambda cid: 0 if cid in future_customers else 1)

print(f"Total Customers Profiled: {len(features):,}")
print("\n--- Target Class Balance ---")
print(features['churn'].value_counts(normalize=True).round(3))
features.head()

### Step 3: Train / Test Split & XGBoost Classifier Training

In [3]:
feature_cols = [
    'recency', 'frequency', 'monetary', 'total_items', 'unique_products',
    'customer_tenure_days', 'avg_order_value', 'max_order_value',
    'std_order_value', 'avg_items_per_order', 'purchase_frequency_rate', 'monthly_spend_rate'
]

X = features[feature_cols]
y = features['churn']

# 80/20 Stratified Split
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, features['Customer ID'], test_size=0.20, random_state=42, stratify=y
)

# Train XGBoost Classifier
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc'
)
model.fit(X_train, y_train)
print("Model training completed successfully!")

### Step 4: Model Evaluation (ROC-AUC, Precision, Recall, Confusion Matrix)

In [4]:
# Predict churn probabilities and binary classes
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"\n=========================================")
print(f"  MODEL PERFORMANCE: ROC-AUC = {auc:.4f}")
print(f"=========================================")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))

print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

### Step 5: Feature Importance (What Drives Churn?)

In [5]:
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- TOP DRIVERS OF CHURN ---")
display(importance_df)

# Calculate high-value at-risk revenue in test set
test_results = pd.DataFrame({
    'CustomerID': id_test,
    'Actual_Churn': y_test,
    'Churn_Probability': y_pred_proba.round(4),
    'Predicted_Churn': y_pred
}).merge(features[['Customer ID', 'monetary', 'recency', 'frequency']], left_on='CustomerID', right_on='Customer ID')

at_risk_rev = test_results[(test_results['Predicted_Churn'] == 1) & (test_results['monetary'] > 1000)]['monetary'].sum()
print(f"\nHigh-Value At-Risk Revenue Identified in Test Set: GBP {at_risk_rev:,.2f}")